# Stage 2: CyTOF preprocessing

Load the split CSVs, apply histone gates, transform marker values, correct batch effects, normalize, and z-score marker values.

In [ ]:
from pathlib import Path
import sys

# Support launching Jupyter from either the checkout root or its notebooks folder.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "app").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
# Prefer this checkout over another installed copy or worktree of app.
sys.path.insert(0, str(PROJECT_ROOT))

# Add the repository's vendored correction and normalization libraries.
sys.path.insert(0, str(PROJECT_ROOT / "external" / "som_ot_warp_repo_anchor"))
sys.path.insert(0, str(PROJECT_ROOT / "external" / "cytof_helper"))
import app  # noqa: E402

# Fail early if Python resolved app from outside this checkout.
if Path(app.__file__).resolve().parent != PROJECT_ROOT / "app":
    raise RuntimeError("app was imported from outside the intended checkout")
print(f"Kernel executable: {sys.executable}")
print(f"Working directory: {Path.cwd()}")
print(f"Imported app: {Path(app.__file__).resolve()}")

In [ ]:
from app.config.config import EXPERIMENTS, METADATA_COLUMNS
from app.config import OUTPUT_ROOT, SKIP_BATCH_CORRECTION, SUBSAMPLE

from app.preprocessing import (
    arcsinh_transform,
    batch_correct,
    gate_histones,
    hierarchical_zscore,
    normalize_pooled_dataframe,
    pool_csv_files,
    subsample_gfp_high,
)

In [ ]:
GFP_INPUT_ROOT = OUTPUT_ROOT / "3. GFP split after CD45 exclusion"
print(f"Output directory: {OUTPUT_ROOT}")
use_labels = False

## Pool and filter GFP-high cells

- Read and concatenate CSV files from the configured GFP folders, preserving each source path in `path`.
- Drop the unused input columns listed in the pooling cell.
- Identify non-metadata columns that contain only missing values in any experiment/sample, then drop those columns from the pooled table.
- Keep only GFP-high cells, using the source folder name to identify the GFP group.

In [ ]:
gfp_csv_files = sorted(
    path for path in GFP_INPUT_ROOT.rglob("*.csv") if path.name != "GFP_split_summary.csv"
)
pooled_gfp_df, missing_columns_by_sample = pool_csv_files(
    gfp_csv_files,
    source_column="path",
    drop_columns=[
        "Line",
        "cell_id",
        "replicate",
        "source_file",
        "source_path",
        "annotation_present",
    ],
    metadata_columns=METADATA_COLUMNS,
)
if not pooled_gfp_df["cell_uid"].is_unique:
    raise ValueError("Pooled cell_uid values must be unique.")


missing_metadata_columns = sorted(set(METADATA_COLUMNS) - set(pooled_gfp_df.columns))
if missing_metadata_columns:
    raise ValueError(f"Pooled input is missing metadata columns: {missing_metadata_columns}")

missing_columns = missing_columns_by_sample.keys()
print("missing Non-metadata columns:", missing_columns)
pooled_gfp_df.drop(missing_columns, inplace=True, axis=1)


pooled_gfp_df["gfp_group"] = pooled_gfp_df["path"].map(lambda path: Path(path).parent.name)
pooled_gfp_df = pooled_gfp_df.copy().loc[pooled_gfp_df["gfp_group"].eq("GFP_high")].copy()
pooled_gfp_df.drop(["gfp_group"], inplace=True, axis=1)
if pooled_gfp_df.empty:
    raise ValueError("No GFP_high cells are available.")

pooled_gfp_df.reset_index(inplace=True)
print("Pooled GFP table shape:", pooled_gfp_df.shape)

## 1. Apply raw histone gates

Keep rows where raw `H3.3` and `H4` both exceed 5. Within each mouse, remove rows with any numeric marker outside the 0.01% to 99.99% quantile bounds, calculated per `sample_name`, `experiment`, and `condition`. `df_gated` contains the retained rows and preserves the input columns.

In [ ]:
df_gated, gated_data_columns = gate_histones(pooled_gfp_df.copy(), skip_columns=METADATA_COLUMNS)
if set(df_gated.columns) != set(gated_data_columns + METADATA_COLUMNS):
    print("ERROR: histone gating changed the dataframe columns.")
print("Pooled gated table shape:", df_gated.shape)

## 2. ArcSinh-transform the gated marker columns

Transform each nonmetadata marker in `df_gated` as `arcsinh(value / 5)`. `df_arcsinh` preserves the input columns and row order.

In [ ]:
df_arcsinh, arcsinh_columns = arcsinh_transform(df_gated.copy(), skip_columns=METADATA_COLUMNS)
if set(df_arcsinh.columns) != set(arcsinh_columns + METADATA_COLUMNS):
    print("ERROR: ArcSinh changed the dataframe columns.")
print("Pooled ArcSinh table shape:", df_arcsinh.shape)

## 3. Apply SOM-OT batch correction

When `SKIP_BATCH_CORRECTION` is true, pass the ArcSinh-transformed table through unchanged. 

Otherwise, fit the SOM-OT model using `df_arcsinh`, then transform its rows using their experiment labels. Shared markers are corrected in one transform call; nonshared or unsupported marker columns are omitted from `df_batch_corrected`.

In [ ]:
if SKIP_BATCH_CORRECTION:
    df_batch_corrected = df_arcsinh
    print("Batch correction skipped")
else:
    df_batch_corrected, corrected_columns = batch_correct(
        df_arcsinh.copy(),
        skip_columns=METADATA_COLUMNS,
    )
    if set(df_batch_corrected.columns) != set(corrected_columns + METADATA_COLUMNS):
        print("ERROR: batch correction changed the dataframe columns.")
    print("Pooled corrected table shape:", df_batch_corrected.shape)

## 4. Normalize marker values

Normalize each input CSV independently using source paths keyed by `cell_uid`. Select every numeric marker except metadata and control markers; all rows are retained in their original order.

In [ ]:
df_normalized, normalized_columns = normalize_pooled_dataframe(
    df_batch_corrected.copy(),
    skip_columns=METADATA_COLUMNS,
)
if not df_normalized.columns.equals(df_batch_corrected.columns):
    print("ERROR: normalization changed the dataframe columns.")
print("Pooled normalized table shape:", df_normalized.shape)

## 5. Z-score marker columns from a balanced reference

- `df_normalized` contains GFP-high cells; no `GFP_high` column is required.
- By default, balance untreated and treated cells across `experiment`, then `sample_name`. Set `use_labels=True` to also balance basal and classical cells; rows missing `labels` are dropped in that mode.
- The function defaults to a maximum reference size of 10,000 cells per group and reduces it when availability requires. It reads the random seed from `RANDOM_STATE` in `.env`.
- Fit one `StandardScaler` on the combined reference and transform every non-metadata feature in the retained rows.
- Return the scaled dataframe, selected and dropped row indices, and a summary with `n` and `n_subsample`; aggregate rows have no `sample_name`.

In [ ]:
df_scaled, scaler, reference_indices, dropped_cell_indices, scaling_summary = hierarchical_zscore(
    df_normalized.copy(),
    use_labels=use_labels,
)
# expected_index = (
#     df_normalized.index[df_normalized["labels"].notna()]
#     if use_labels
#     else df_normalized.index
# )
# if not df_scaled.columns.equals(df_normalized.columns) or not df_scaled.index.equals(expected_index):
#     raise ValueError("Reference scaling changed the expected input rows or columns.")


print(f"Reference cells selected: {len(reference_indices)}")
print(f"Cells dropped for missing labels: {len(dropped_cell_indices)}")
print("Pooled z-scored table shape:", df_scaled.shape)
display(scaling_summary)

## 6. Subsample and save data

- `df_scaled` already contains GFP-high cells.
- When `SUBSAMPLE` is true, select equal sample sizes within each experiment, balancing by condition and sample; `use_labels=True` also balances subtype labels.
- Build the summary with the same grouping mode, including per-sample counts and aggregate counts without `sample_name`.
- Save the selected data, summary, and complete z-scored input.

In [ ]:
import pandas as pd

df, summary = subsample_gfp_high(
    df_scaled,
    subsample=True,
    use_labels=True,
)
GFP_HIGH_SUFFIX = "_BALANCED" if SUBSAMPLE else ""
GFP_HIGH_OUT = OUTPUT_ROOT / "6. pooled_GFP_high_ZSCORED"
GFP_HIGH_OUT.mkdir(parents=True, exist_ok=True)
print(f"Subsampling enabled: {SUBSAMPLE}")
print(f"Label balancing enabled: {use_labels}")
print("Selected z-scored GFP_high table shape:", df.shape)
with pd.option_context("display.max_rows", None):
    display(summary)

df.to_csv(GFP_HIGH_OUT / f"all_GFP_high_ZSCORED{GFP_HIGH_SUFFIX}.csv", index=False)
summary.to_csv(GFP_HIGH_OUT / f"GFP_high_selection_summary{GFP_HIGH_SUFFIX}.csv", index=False)
df_scaled.to_csv(GFP_HIGH_OUT / "all_GFP_high_and_GFP_low_ZSCORED.csv", index=False)